<a href="https://colab.research.google.com/github/abduyea/Career-Trends-Analyzer/blob/main/notebooks/Proejct_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Abdulfetah Adem

Spencer K

#Project Objective

Develop a data-driven tool that identifies the most in-demand technical and professional skills using real-world job posting data.

The tool will highlight skill gaps and provide an interactive dashboard to explore trends across industries, regions, and time periods.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

from google.colab import drive

# mount Drive
drive.mount("/content/drive", force_remount=False)

# core paths
ROOT_DIR = Path("/content/drive/MyDrive/Career-Trends-Analyzer")
DATA_DIR = ROOT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
SRC_DIR = ROOT_DIR / "src"

# ensure folders
for path in (DATA_DIR, RAW_DIR, SRC_DIR):
    path.mkdir(parents=True, exist_ok=True)

# make project importable
root_str = str(ROOT_DIR)
if root_str not in sys.path:
    sys.path.append(root_str)

# src as a package
init_code = '__all__ = ["config", "data_loader", "utils"]\n'
(SRC_DIR / "__init__.py").write_text(init_code, encoding="utf-8")

# write config.py
config_code = """
from __future__ import annotations

from pathlib import Path

ROOT_DIR = Path("/content/drive/MyDrive/Career-Trends-Analyzer")
DATA_DIR = ROOT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"

COMP_DIR = RAW_DIR / "companies"
JOBS_DIR = RAW_DIR / "jobs"
MAP_DIR = RAW_DIR / "mappings"
""".strip() + "\n"

(SRC_DIR / "config.py").write_text(config_code, encoding="utf-8")

print("Setup OK")
print("ROOT_DIR:", ROOT_DIR)
print("SRC_DIR :", SRC_DIR)


Mounted at /content/drive
Setup OK
ROOT_DIR: /content/drive/MyDrive/Career-Trends-Analyzer
SRC_DIR : /content/drive/MyDrive/Career-Trends-Analyzer/src


In [3]:
from pathlib import Path

SRC_DIR = Path("/content/drive/MyDrive/Career-Trends-Analyzer/src")

# ---------------- data_loader.py ----------------
data_loader_code = """
from __future__ import annotations

from pathlib import Path
from typing import Dict

import pandas as pd

from .config import RAW_DIR

try:
    from google.colab import drive
except ImportError:
    drive = None


def mount_drive() -> None:
    if drive:
        drive.mount("/content/drive", force_remount=False)


def _load_csv(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f"CSV not found: {path}")
    return pd.read_csv(path)


def load_postings() -> pd.DataFrame:
    return _load_csv(RAW_DIR / "postings.csv")


def _load_folder(name: str) -> Dict[str, pd.DataFrame]:
    folder = RAW_DIR / name
    if not folder.is_dir():
        return {}
    return {p.stem: pd.read_csv(p) for p in folder.glob("*.csv")}


def load_companies() -> Dict[str, pd.DataFrame]:
    return _load_folder("companies")


def load_jobs() -> Dict[str, pd.DataFrame]:
    return _load_folder("jobs")


def load_mappings() -> Dict[str, pd.DataFrame]:
    return _load_folder("mappings")


def build_master(
    postings: pd.DataFrame,
    companies: Dict[str, pd.DataFrame],
    jobs: Dict[str, pd.DataFrame],
    mappings: Dict[str, pd.DataFrame],
) -> pd.DataFrame:
    df = postings.copy()

    c = companies.get("companies")
    if c is not None and "company_id" in df and "company_id" in c:
        df = df.merge(c, on="company_id", how="left", suffixes=("", "_company"))

    s = jobs.get("salaries")
    if s is not None and "job_id" in df and "job_id" in s:
        df = df.merge(s, on="job_id", how="left", suffixes=("", "_salary"))

    return df
""".strip() + "\n"

(SRC_DIR / "data_loader.py").write_text(data_loader_code, encoding="utf-8")

# ---------------- utils.py ----------------
utils_code = """
from __future__ import annotations

from typing import Dict

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print


def peek(df: pd.DataFrame, n: int = 5) -> None:
    print(df.shape)
    display(df.head(n))


def missing_summary(df: pd.DataFrame, n: int = 20) -> None:
    miss = df.isna().sum().sort_values(ascending=False)
    print(miss.head(n))


def summarize_tables(tables: Dict[str, pd.DataFrame]) -> None:
    for name, frame in tables.items():
        print(f"{name}: {frame.shape}")
""".strip() + "\n"

(SRC_DIR / "utils.py").write_text(utils_code, encoding="utf-8")

print("data_loader.py and utils.py written in:", SRC_DIR)


data_loader.py and utils.py written in: /content/drive/MyDrive/Career-Trends-Analyzer/src
